# 03 — Medir Y y graficar scorecard

**Track:** `docs/experiment/RESEARCH_TRACK.md` · **Oleada 3**

Lee artefactos DoE/seed y grafica Y **desagregadas** (recall, AC@k, latency).
`composite_score` es secundario — no usarlo como veredicto.

Ejecutar tras NB-02 o con reports seed existentes.

In [ ]:
from pathlib import Path
import json
import shutil

import pandas as pd
import matplotlib.pyplot as plt

REPO = Path("../../..").resolve()
REPORTS = REPO / "benchmarks/domains/knowledge_graphs/reports"
OUT = REPORTS / "research"
OUT.mkdir(parents=True, exist_ok=True)

results_csv = REPORTS / "results.csv"
if not results_csv.is_file():
    raise FileNotFoundError("Run NB-02 or offline DoE first: " + str(results_csv))

# Copia de trabajo (no pisar seed)
shutil.copy2(results_csv, OUT / "results_snapshot.csv")
df = pd.read_csv(results_csv)
df.head()

In [ ]:
print("columns:", list(df.columns))
y_candidates = [
    c for c in df.columns
    if any(k in c.lower() for k in ("recall", "answer", "latency", "evidence", "correctness"))
]
y_candidates

In [ ]:
factor_cols = [c for c in df.columns if c.lower() in {"inference", "chunk_size", "chunking", "rag"}]
plot_cols = y_candidates[:3] or [c for c in df.select_dtypes("number").columns][:3]

if factor_cols and plot_cols:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(4 * len(plot_cols), 3), squeeze=False)
    for ax, col in zip(axes[0], plot_cols):
        grp = df.groupby(factor_cols[0])[col].mean()
        grp.plot(kind="bar", ax=ax, title=col)
        ax.set_xlabel(factor_cols[0])
    fig.tight_layout()
    fig.savefig(OUT / "03_y_by_factor.png", dpi=120)
    plt.show()
else:
    df.describe()

In [ ]:
# Veredictos seed (solo lectura)
for name in ("hi_wave_verdict.json", "pipeline_closure.json", "family_wave_verdict.json"):
    p = REPORTS / name
    if p.is_file():
        print(name, json.loads(p.read_text(encoding="utf-8")))

## Lectura

- Gate seed PASS ≠ PRODUCT §5.
- Si AC@k está saturado (~1.0), endurecer probes (oleada 5) antes de afirmar H_chunk en tarea.